In [1]:
import rpy2.robjects.packages as rpackages
import rpy2.robjects as ro
from rpy2.robjects import pandas2ri
import pandas as pd
pandas2ri.activate()

rpackages.importr('DBI')
rpackages.importr('lme4')
rpackages.importr('DT')

rpy2.robjects.packages.Package as a <module 'DT'>

In [2]:
filter_model_id = 5445338
ro.r(f'''
           RESULTS_DB_PATH <- "/home/abishekthamma/PycharmProjects/masters_thesis/ss-llm/nanoGPT/results/results.db"
results_db <- dbConnect(RSQLite::SQLite(), RESULTS_DB_PATH)

baseline_df <- dbGetQuery(results_db,"
SELECT SPRTNaturalStories.RTUID, 
SPRTNaturalStories.WorkerID, 
SPRTNaturalStories.StoryWordID, 
SPRTNaturalStories.RT, 
WordDetails.Word as WordCategory, 
WordDetails.CharacterLength, 
WordDetails.WordUID as WordCategoryID,
WordDetails.LogFrequencies as LogFrequencies,
Story.POSTag as POSTag
FROM SPRTNaturalStories 
JOIN Story on SPRTNaturalStories.StoryWordID = Story.StoryWordID 
JOIN WordDetails on WordDetails.WordUID = Story.WordUID 

     
")
'''
)
ro.r("baseline_df")

R[write to console]: In addition: 
R[write to console]: Warning messages:

R[write to console]: 1: 
R[write to console]: In (function (package, help, pos = 2, lib.loc = NULL, character.only = FALSE,  :
R[write to console]: 
 
R[write to console]:  library ‘/usr/lib/R/site-library’ contains no packages

R[write to console]: 2: 
R[write to console]: In (function (package, help, pos = 2, lib.loc = NULL, character.only = FALSE,  :
R[write to console]: 
 
R[write to console]:  library ‘/usr/lib/R/site-library’ contains no packages

R[write to console]: 3: 
R[write to console]: In (function (package, help, pos = 2, lib.loc = NULL, character.only = FALSE,  :
R[write to console]: 
 
R[write to console]:  library ‘/usr/lib/R/site-library’ contains no packages



RTUID,WorkerID,StoryWordID,...,WordCategoryID,LogFrequencies,POSTag
1,...,1,...,1,...,...
2,,1,,1,,
3,,1,,1,,
4,,1,,1,,
...,,...,,...,,
848764,,10256,,2141,,
848765,,10256,,2141,,
848766,,10256,,2141,,
848767,,10256,,2141,,


In [4]:
ro.r('''
baseline_df$LogRT <- log(baseline_df$RT)

baseline_df$WorkerID <- as.factor(baseline_df$WorkerID)
baseline_df$WordCategoryID <- as.factor(baseline_df$WordCategoryID)
baseline_df$POSTag <- as.factor(baseline_df$POSTag)
baseline_df$CharacterLength_c <- scale(baseline_df$CharacterLength)

#Should I scale Log Frequencies(?)
baseline_df$LogFrequencies_c <- scale(baseline_df$LogFrequencies)

''')

In [5]:
ro.r("baseline_df")

RTUID,WorkerID,StoryWordID,...,LogRT,CharacterLength_c,LogFrequencies_c
1,A3QJP...,1,...,...,...,...
2,A2RPQ...,1,,,,
3,A11KM...,1,,,,
4,A1U1Q...,1,,,,
...,...,...,,,,
848764,A253Q...,10256,,,,
848765,A1WUR...,10256,,,,
848766,A1INW...,10256,,,,
848767,A279T...,10256,,,,


In [35]:
ro.r('''
exp_index <- sample(1:nrow(baseline_df), nrow(baseline_df)*0.5)

baseline_df$exp <- 0
baseline_df$exp[exp_index] <- 1
baseline_df$exp <- as.factor(baseline_df$exp)

exploratory_df <- baseline_df[baseline_df$exp == 1,]
test_df <- baseline_df[baseline_df$exp == 0,]
''')

In [2]:

def extract_model_summary(model_name, summary_name):

    # 1. Fixed Effects
    fixed_effects = ro.r(f'as.data.frame({summary_name}$coefficients)')
    fixed_effects_df = pandas2ri.rpy2py(fixed_effects)
    #print(fixed_effects_df)
    fixed_effects_df.columns = ['Estimate', 'Std. Error', 't value'] #, 'Pr(>|t|)']

    # 2. Random Effects
    random_effects = ro.r(f'as.data.frame(VarCorr({model_name}))')
    random_effects_df = pandas2ri.rpy2py(random_effects)
    #print("\nRandom Effects (Variance and Std. Dev by Group):\n", random_effects_df)

    # 3. Residuals
    residuals = ro.r(f'as.data.frame({summary_name}$residuals)')
    residuals_df = pandas2ri.rpy2py(residuals)
    #print("\nResiduals:\n", residuals_df)

    # 4. Model Fit Statistics
    aic = ro.r(f'AIC({model_name})')[0]
    bic = ro.r(f'BIC({model_name})')[0]
    log_likelihood = ro.r(f'logLik({model_name})')[0]
    warnings = ro.r('warnings()') 
    fit_stats_df = pd.DataFrame({
        'AIC': [aic],
        'BIC': [bic],
        'Log-Likelihood': [log_likelihood],
        'Warnings': [warnings]
    })

    # 5. Variance-Covariance Matrix of Random Effects
    var_cov_matrix = ro.r(f'as.data.frame({summary_name}$varcor)')
    var_cov_matrix_df = pandas2ri.rpy2py(var_cov_matrix)
    
    return fixed_effects_df, random_effects_df, fit_stats_df, var_cov_matrix_df

def execute_r_lmer_model(fit_formula, data_frame_name, fit_name, fit_summary_name):
    ro.r(f'''
    {fit_name} <- lmer({fit_formula}, data={data_frame_name}, REML=F)
    {fit_summary_name} <- summary({fit_name})
    ''')
    
    fixed_effects, random_effects, fit_stats, var_cov_matrix = extract_model_summary(fit_name, fit_summary_name)

    return fixed_effects, random_effects, fit_stats, var_cov_matrix


#Given a model id, load the dataset for it in r, transform the data and run the model and return the results

def fit_surprisal_model(filter_model_id):
    ro.r(f'''
           RESULTS_DB_PATH <- "/home/abishekthamma/PycharmProjects/masters_thesis/ss-llm/nanoGPT/results/results.db"
            results_db <- dbConnect(RSQLite::SQLite(), RESULTS_DB_PATH)

            baseline_df <- dbGetQuery(results_db,"WITH FilteredModel AS (
            SELECT StoryWordID, SurprisalScore
            FROM ModelSurprisalScores
            WHERE ModelID = {filter_model_id}
            )

            SELECT SPRTNaturalStories.RTUID, 
            SPRTNaturalStories.WorkerID, 
            SPRTNaturalStories.StoryWordID, 
            SPRTNaturalStories.RT, 
            WordDetails.Word as WordCategory, 
            WordDetails.CharacterLength, 
            WordDetails.WordUID as WordCategoryID,
            WordDetails.LogFrequencies as LogFrequencies,
            Story.POSTag as POSTag,
            FilteredModel.SurprisalScore as SurprisalScore
            FROM SPRTNaturalStories 
            JOIN Story on SPRTNaturalStories.StoryWordID = Story.StoryWordID 
            JOIN WordDetails on WordDetails.WordUID = Story.WordUID 
            JOIN FilteredModel on FilteredModel.StoryWordID = SPRTNaturalStories.StoryWordID

                
            ")
            '''
            )
    
    
    ro.r('''
        baseline_df$LogRT <- log(baseline_df$RT)

        baseline_df$WorkerID <- as.factor(baseline_df$WorkerID)
        baseline_df$WordCategoryID <- as.factor(baseline_df$WordCategoryID)
        baseline_df$POSTag <- as.factor(baseline_df$POSTag)
        baseline_df$CharacterLength_c <- scale(baseline_df$CharacterLength)

        #Should I scale Log Frequencies(?)
        baseline_df$LogFrequencies_c <- scale(baseline_df$LogFrequencies)

        baseline_df$SurprisalScore_c <- scale(baseline_df$SurprisalScore)

        ''')
    
    data_frame_name = "baseline_df"

    fit_name_m = "fit_model"
    fit_summary_name_m = "fit_model_summary"
    fit_formula_m = "LogRT ~ CharacterLength_c + LogFrequencies_c + SurprisalScore_c + (1 | WorkerID) + (1 | POSTag)"

    fixed_effects_m, random_effects_m, fit_stats_m, var_cov_matrix_m = execute_r_lmer_model(fit_formula_m, "baseline_df", fit_name_m, fit_summary_name_m)

    return fixed_effects_m, random_effects_m, fit_stats_m, var_cov_matrix_m

def compile_results_as_df(model_id, fixed_effects_df, random_effects_df, fit_stats_df, var_cov_matrix_df, fit_stats_b):
    
    results_df_row = {
        "ModelID" : model_id,
        "Log-Likelihood" : fit_stats_df['Log-Likelihood'].values[0],
        "Coefficient for Surprisal Score" : fixed_effects_df.loc['SurprisalScore_c', 'Estimate'],
        "Delta Log-Likelihood" : fit_stats_df['Log-Likelihood'].values[0] - fit_stats_b['Log-Likelihood'].values[0],
        "AIC" : fit_stats_df['AIC'].values[0],
        "BIC" : fit_stats_df['BIC'].values[0],
        'Fixed Effects': fixed_effects_df.to_html(),
        'Random Effects': random_effects_df.to_html(),
        'Variance-Covariance Matrix': var_cov_matrix_df.to_html(),
    }
        
    #results_df = pd.DataFrame(results_df_row, index=[0])

    return results_df_row


import pandas as pd

def append_or_overwrite_csv(file_path, new_row):
    # Read the existing CSV file
    try:
        df = pd.read_csv(file_path)
    except FileNotFoundError:
        # If the file does not exist, create a new DataFrame
        df = pd.DataFrame(columns=new_row.keys())
    
    # Check if the row already exists
    # This assumes that the DataFrame has a unique identifier column called 'id'
    if 'ModelID' in new_row:  # Ensure that there's a unique identifier
        existing_row_index = df[df['ModelID'] == new_row['ModelID']].index
        
        if not existing_row_index.empty:
            # If the row exists, update it
            df.loc[existing_row_index[0]] = new_row  # Update the first matching row
        else:
            # If the row does not exist, append it
            df = pd.concat([df, pd.DataFrame([new_row])], ignore_index=True)
    else:
        print("New row must contain a unique identifier under the key 'ModelID'.")

    # Save the updated DataFrame back to CSV
    df.to_csv(file_path, index=False)



In [3]:






# fit_name_m = "fit_model"
# fit_summary_name_m = "fit_summary_model"
# fit_formula_m = "LogRT ~ CharacterLength_c + LogFrequencies_c + SurprisalScore_c + (1 | WorkerID) + (1 | POSTag)"
# fixed_effects_m, random_effects_m, fit_stats_m, var_cov_matrix_m = execute_r_lmer_model(fit_formula_m, data_frame_name, fit_name_m, fit_summary_name_m)

# delta_log_likelihood = fit_stats_m['Log-Likelihood'].values[0] - fit_stats_b['Log-Likelihood'].values[0]


# print("\nDelta Log-Likelihood:", delta_log_likelihood)
# print("\nBaseline Model:")
# print(fixed_effects_b)
# print(random_effects_b)
# print(fit_stats_b)
# print(var_cov_matrix_b)

# print("\nModel:")
# print(fixed_effects_m)
# print(random_effects_m)
# print(fit_stats_m)
# print(var_cov_matrix_m)

# print("Model coefficient for surprisal score: ", fixed_effects_m.loc['SurprisalScore_c', 'Estimate'])

In [4]:
#Given a set of details, return a dataframe compiled with the details

ro.r(f'''RESULTS_DB_PATH <- "/home/abishekthamma/PycharmProjects/masters_thesis/ss-llm/nanoGPT/results/results.db"
        results_db <- dbConnect(RSQLite::SQLite(), RESULTS_DB_PATH)
        baseline_df <- dbGetQuery(results_db,"
        SELECT SPRTNaturalStories.RTUID, 
        SPRTNaturalStories.WorkerID, 
        SPRTNaturalStories.StoryWordID, 
        SPRTNaturalStories.RT, 
        WordDetails.Word as WordCategory, 
        WordDetails.CharacterLength, 
        WordDetails.WordUID as WordCategoryID,
        WordDetails.LogFrequencies as LogFrequencies,
        Story.POSTag as POSTag
        FROM SPRTNaturalStories 
        JOIN Story on SPRTNaturalStories.StoryWordID = Story.StoryWordID 
        JOIN WordDetails on WordDetails.WordUID = Story.WordUID                 
        ")
            '''
            )

    
ro.r('''
        baseline_df$LogRT <- log(baseline_df$RT)

        baseline_df$WorkerID <- as.factor(baseline_df$WorkerID)
        baseline_df$WordCategoryID <- as.factor(baseline_df$WordCategoryID)
        baseline_df$POSTag <- as.factor(baseline_df$POSTag)
        baseline_df$CharacterLength_c <- scale(baseline_df$CharacterLength)

        #Should I scale Log Frequencies(?)
        baseline_df$LogFrequencies_c <- scale(baseline_df$LogFrequencies)

        ''')


data_frame_name = "baseline_df"
fit_name_b = "fit_baseline"
fit_summary_name_b = "fit_summary_baseline"
fit_formula_b = "LogRT ~ CharacterLength_c + LogFrequencies_c + (1 | WorkerID) + (1 | POSTag)"
fixed_effects_b, random_effects_b, fit_stats_b, var_cov_matrix_b = execute_r_lmer_model(fit_formula_b, data_frame_name, fit_name_b, fit_summary_name_b)




R[write to console]: In addition: 
R[write to console]: Warning messages:

R[write to console]: 1: 
R[write to console]: In (function (package, help, pos = 2, lib.loc = NULL, character.only = FALSE,  :
R[write to console]: 
 
R[write to console]:  library ‘/usr/lib/R/site-library’ contains no packages

R[write to console]: 2: 
R[write to console]: In (function (package, help, pos = 2, lib.loc = NULL, character.only = FALSE,  :
R[write to console]: 
 
R[write to console]:  library ‘/usr/lib/R/site-library’ contains no packages

R[write to console]: 3: 
R[write to console]: In (function (package, help, pos = 2, lib.loc = NULL, character.only = FALSE,  :
R[write to console]: 
 
R[write to console]:  library ‘/usr/lib/R/site-library’ contains no packages



In [5]:
import os
import sqlite3
import platform

if "pop-os" in platform.node():
    ROOT = r"/home/abishekthamma/PycharmProjects/masters_thesis/ss-llm/nanoGPT/"
else:
    ROOT = r'/gpfs/home4/athamma/repo/ss-llm/nanoGPT/'
    
TOKENIZER_ROOT = os.path.join(ROOT, "data")
OUT_ROOT = os.path.join(ROOT, "output_dump")
RESULTS_ROOT = os.path.join(ROOT, "results")

SQL_DB = os.path.join(RESULTS_ROOT, "results.db")


def create_connection_cursor(db_file):
    """
    Create a database connection to the SQLite database specified by the db_file

    Args:
        db_file (str): database file

    Returns:
        Connection object or None
    """
    conn = sqlite3.connect(db_file)
    c = conn.cursor()
    return conn, c

conn, c = create_connection_cursor(SQL_DB)


model_id_list = pd.read_sql_query("SELECT ModelID FROM ModelSurprisalScores", conn)['ModelID'].unique().tolist()
print(len(model_id_list), model_id_list)



233 [5444724, 5445338, 5492054, 5492134, 5496426, 5496427, 5734459, 5734464, 5734465, 5734467, 5734550, 5757736, 5757737, 5983308, 5983309, 5988018, 5988019, 5988020, 5988022, 5989080, 5989082, 6486043, 6486044, 6603578, 6603579, 6603580, 6607670, 6620547, 6620548, 6621801, 6681922, 6681937, 6681938, 6681939, 6681940, 6681941, 6681942, 6681944, 6681945, 6681946, 6681947, 6681948, 6681949, 6681950, 6681951, 6683308, 6683309, 6683310, 6683311, 6689751, 6689752, 6689753, 6810203, 6810205, 6810296, 6810297, 6810320, 6810321, 6810323, 6810325, 6810326, 6839399, 6839402, 6839403, 6839404, 6839405, 6839409, 6839410, 6839411, 6839412, 6839413, 6839414, 6839416, 6839417, 6839418, 6839419, 6839420, 6839421, 6839422, 6839423, 6839424, 6839425, 6839426, 6839427, 6839428, 6839429, 6839430, 6839431, 6849723, 6849725, 6864685, 6890225, 6890228, 6890229, 6890230, 6890231, 6890232, 6890233, 6890234, 6890235, 6890236, 6890237, 6890238, 6890239, 6890240, 6890241, 6890242, 6890243, 6890244, 6890245, 68902

In [6]:
import tqdm

write_path = "surprisal_analysis_results_temp.csv"

#results_df = pd.DataFrame()

def verify_model_already_processed(model_id, write_path):
    try:
        df = pd.read_csv(write_path)
        return df['ModelID'].tolist()
    except FileNotFoundError:
        return []

    if model_id in df['ModelID'].tolist():
        return True
    else:
        return False
    

for model_id in tqdm.tqdm(model_id_list):
    if model_id in verify_model_already_processed(model_id, write_path):
        continue
    try:    
        fixed_effects_m, random_effects_m, fit_stats_m, var_cov_matrix_m = fit_surprisal_model(model_id)
        append_row = compile_results_as_df(model_id, fixed_effects_m, random_effects_m, fit_stats_m, var_cov_matrix_m, fit_stats_b)
        append_or_overwrite_csv(write_path, append_row)
    except Exception as e:
        print(f"Error for model id {model_id}: {e}")
        continue
        



  0%|          | 0/233 [00:00<?, ?it/s]

 73%|███████▎  | 170/233 [00:19<00:00, 180.86it/s]R[write to console]: In addition: 
R[write to console]: Warning message:

R[write to console]: call dbDisconnect() when finished working with a connection 

 76%|███████▌  | 177/233 [00:59<01:21,  1.45s/it] R[write to console]: In addition: 
R[write to console]: Warning message:

R[write to console]: In checkConv(attr(opt, "derivs"), opt$par, ctrl = control$checkConv,  :
R[write to console]: 
 
R[write to console]:  Model failed to converge with max|grad| = 0.00382132 (tol = 0.002, component 1)

 79%|███████▉  | 184/233 [03:58<11:05, 13.58s/it]R[write to console]: In addition: 
R[write to console]: Warning message:

R[write to console]: In checkConv(attr(opt, "derivs"), opt$par, ctrl = control$checkConv,  :
R[write to console]: 
 
R[write to console]:  Model failed to converge with max|grad| = 0.00240113 (tol = 0.002, component 1)

100%|██████████| 233/233 [07:05<00:00,  1.83s/it]


In [ ]:

results_df

,Log-Likelihood,Coefficient for Surprisal Score,Delta Log-Likelihood,AIC,BIC,Fixed Effects,Random Effects,Variance-Covariance Matrix
0,-169650.372658,0.025981,1721.849012,339314.745315,339396.306095,"<table border=""1"" class=""dataframe"">\n <thead...","<table border=""1"" class=""dataframe"">\n <thead...","<table border=""1"" class=""dataframe"">\n <thead..."
0,-169644.225553,0.025955,1727.996117,339302.451107,339384.011887,"<table border=""1"" class=""dataframe"">\n <thead...","<table border=""1"" class=""dataframe"">\n <thead...","<table border=""1"" class=""dataframe"">\n <thead..."


137 [5444724, 5445338, 5492054, 5492134, 5496426, 5496427, 5734459, 5734464, 5734465, 5734467, 5734550, 5757736, 5757737, 5983308, 5983309, 5988018, 5988019, 5988020, 5988022, 5989080, 5989082, 6486043, 6486044, 6603578, 6603579, 6603580, 6607670, 6620547, 6620548, 6621801, 6681922, 6681937, 6681938, 6681939, 6681940, 6681941, 6681942, 6681944, 6681945, 6681946, 6681947, 6681948, 6681949, 6681950, 6681951, 6683308, 6683309, 6683310, 6683311, 6689751, 6689752, 6689753, 6810203, 6810205, 6810296, 6810297, 6810320, 6810321, 6810323, 6810325, 6810326, 6839399, 6839402, 6839403, 6839404, 6839405, 6839409, 6839410, 6839411, 6839412, 6839413, 6839414, 6839416, 6839417, 6839418, 6839419, 6839420, 6839421, 6839422, 6839423, 6839424, 6839425, 6839426, 6839427, 6839428, 6839429, 6839430, 6839431, 6849723, 6849725, 6864685, 6890225, 6890228, 6890229, 6890230, 6890231, 6890232, 6890233, 6890234, 6890235, 6890236, 6890237, 6890238, 6890239, 6890240, 6890241, 6890242, 6890243, 6890244, 6890245, 68902

In [ ]:
fit_formula_list = [
    # "LogRT ~ CharacterLength_c  + ( 1 | WorkerID) + (1 | WordCategoryID)",
    # "LogRT ~ CharacterLength_c  + ( 1 | WorkerID) + (1 | POSTag)",
    
    # "LogRT ~ LogFrequencies_c + ( 1 | WorkerID) + (1 | WordCategoryID)",
    # "LogRT ~ LogFrequencies_c + ( 1 | WorkerID) + (1 | POSTag)",
    
    # "LogRT ~ CharacterLength_c + LogFrequencies_c + ( 1 | WorkerID) + (1 | WordCategoryID)",
    # "LogRT ~ CharacterLength_c + LogFrequencies_c + ( 1 | WorkerID) + (1 | POSTag)",
    
    # "LogRT ~ CharacterLength_c + LogFrequencies_c + ( 1 + CharacterLength_c | WorkerID) + (1 | WordCategoryID)",
    # "LogRT ~ CharacterLength_c + LogFrequencies_c + ( 1 + CharacterLength_c | WorkerID) + (1 | POSTag)",

    # "LogRT ~ CharacterLength_c + LogFrequencies_c + ( 1 + LogFrequencies_c | WorkerID) + (1 | WordCategoryID)",
    # "LogRT ~ CharacterLength_c + LogFrequencies_c + ( 1 + LogFrequencies_c | WorkerID) + (1 | POSTag)",

    # "LogRT ~ CharacterLength_c + LogFrequencies_c + ( 1 + CharacterLength_c + LogFrequencies_c | WorkerID) + (1 | WordCategoryID)",
    # "LogRT ~ 1 + ( 1 + CharacterLength_c + LogFrequencies_c | WorkerID) + (1 | WordCategoryID)",    
    # "LogRT ~ 1 + ( 1 + CharacterLength_c + LogFrequencies_c | WorkerID) + (1 | POSTag)",
    # "LogRT ~ CharacterLength_c + LogFrequencies_c + ( 1 + CharacterLength_c + LogFrequencies_c | WorkerID) + (1 | POSTag)",


]

In [138]:
from tqdm import tqdm
import time
results_list = []

for fit_formula in tqdm(fit_formula_list):
    start_time = time.time()
    fixed_effects, random_effects, fit_stats, var_cov_matrix = execute_r_lmer_model(fit_formula, data_frame_name, fit_name, fit_summary_name)
    
    results_list.append({"fit_formula": fit_formula, 
                         "fixed_effects": fixed_effects, 
                         "random_effects": random_effects, 
                         "fit_stats": fit_stats,
                         "var_cov_matrix": var_cov_matrix,
                         "execution_time": time.time() - start_time})


results_list

  0%|          | 0/3 [00:00<?, ?it/s]R[write to console]: In addition: 
R[write to console]: Warning message:

R[write to console]: In checkConv(attr(opt, "derivs"), opt$par, ctrl = control$checkConv,  :
R[write to console]: 
 
R[write to console]:  Model failed to converge with max|grad| = 0.00397905 (tol = 0.002, component 1)

 33%|███▎      | 1/3 [05:35<11:10, 335.21s/it]R[write to console]: In addition: 
R[write to console]: Warning message:

R[write to console]: In checkConv(attr(opt, "derivs"), opt$par, ctrl = control$checkConv,  :
R[write to console]: 
 
R[write to console]:  Model failed to converge with max|grad| = 0.0128911 (tol = 0.002, component 1)

100%|██████████| 3/3 [10:28<00:00, 209.60s/it]


[{'fit_formula': 'LogRT ~ 1 + ( 1 + CharacterLength_c + LogFrequencies_c | WorkerID) + (1 | WordCategoryID)',
  'fixed_effects':              Estimate  Std. Error     t value
  (Intercept)   5.61817    0.019124  293.779003,
  'random_effects':               grp               var1               var2      vcov     sdcor
  1  WordCategoryID        (Intercept)               None  0.006672  0.081683
  2        WorkerID        (Intercept)               None  0.071306  0.267033
  3        WorkerID  CharacterLength_c               None  0.000372  0.019278
  4        WorkerID   LogFrequencies_c               None  0.000555  0.023568
  5        WorkerID        (Intercept)  CharacterLength_c  0.002026  0.393496
  6        WorkerID        (Intercept)   LogFrequencies_c -0.003187 -0.506457
  7        WorkerID  CharacterLength_c   LogFrequencies_c -0.000300 -0.659293
  8        Residual               None               None  0.084373  0.290470,
  'fit_stats':              AIC            BIC  Log-Lik

In [151]:
ro.r('warnings()')

ValueError: Not an R object.

<rpy2.robjects.vectors.ListVector object at 0x7eadc57b8740> [RTYPES.VECSXP]
R classes: ('warnings',)
[LangSexpVector]
  Model failed to converge with max|grad| = 0.0128911 (tol = 0.002, component 1): <class 'rpy2.rinterface.LangSexpVector'>
  <rpy2.rinterface.LangSexpVector object at 0x7eadb1da24c0> [RTYPES.LANGSXP]

In [141]:
len(results_list)

14

In [157]:
results_df = pd.DataFrame(results_list)
results_df["AIC"] = results_df["fit_stats"].apply(lambda x: x["AIC"].values[0])
results_df["BIC"] = results_df["fit_stats"].apply(lambda x: x["BIC"].values[0])
results_df["Log-Likelihood"] = results_df["fit_stats"].apply(lambda x: x["Log-Likelihood"].values[0])

In [158]:
results_df = results_df.drop(columns=["fit_stats"])
results_df

,fit_formula,fixed_effects,random_effects,var_cov_matrix,execution_time,AIC,BIC,Log-Likelihood
0,LogRT ~ 1 + ( 1 + CharacterLength_c + LogFrequ...,Estimate Std. Error t value ...,grp var1 ...,grp var1 ...,335.209563,161687.121799,161785.747324,-80834.560900
1,LogRT ~ 1 + ( 1 + CharacterLength_c + LogFrequ...,Estimate Std. Error t value (I...,grp var1 v...,grp var1 v...,115.139529,169040.543532,169139.169057,-84511.271766
2,LogRT ~ CharacterLength_c + LogFrequencies_c +...,Estimate Std. Error t ...,grp var1 v...,grp var1 v...,178.420686,168816.997152,168937.539460,-84397.498576
3,LogRT ~ CharacterLength_c + ( 1 | WorkerID) +...,Estimate Std. Error t ...,grp var1 var2 vcov...,grp var1 var2 vcov...,15.779711,163426.866042,163481.658000,-81708.433021
4,LogRT ~ CharacterLength_c + ( 1 | WorkerID) +...,Estimate Std. Error t ...,grp var1 var2 vcov s...,grp var1 var2 vcov s...,11.176801,171695.944476,171750.736434,-85842.972238
5,LogRT ~ LogFrequencies_c + ( 1 | WorkerID) + (...,Estimate Std. Error t v...,grp var1 var2 vcov...,grp var1 var2 vcov...,18.168842,163314.069746,163368.861704,-81652.034873
6,LogRT ~ LogFrequencies_c + ( 1 | WorkerID) + (...,Estimate Std. Error t v...,grp var1 var2 vcov s...,grp var1 var2 vcov s...,11.620378,170939.632793,170994.424751,-85464.816397
7,LogRT ~ CharacterLength_c + LogFrequencies_c +...,Estimate Std. Error t ...,grp var1 var2 vcov...,grp var1 var2 vcov...,20.015982,163253.665880,163319.416230,-81620.832940
8,LogRT ~ CharacterLength_c + LogFrequencies_c +...,Estimate Std. Error t ...,grp var1 var2 vcov s...,grp var1 var2 vcov s...,13.127882,170658.376033,170724.126383,-85323.188017
9,LogRT ~ CharacterLength_c + LogFrequencies_c +...,Estimate Std. Error t ...,grp var1 ...,grp var1 ...,86.378951,161711.526039,161799.193172,-80847.763020
